In [ ]:
from pathlib import Path
import gc
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from IPython.display import display
from matplotlib.ticker import FuncFormatter
from PIL import Image
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms


# ============================================================
# 1. Konfigurasi
# ============================================================

# Jalur lokal Windows.
IMAGE_DIR = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\gambar_terpilih"
)
CSV_PATH = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv"
    r"\gabungan_dengan_jumlah_sel.csv"
)
OUTPUT_DIR = Path(r"D:\00 Kuliah\TA\Experiment\Model\5 DL\Hasil\Chronological Order")

# Jika berjalan di Kaggle dan path Windows tidak tersedia,
# cari folder/file berdasarkan namanya di /kaggle/input.
KAGGLE_INPUT = Path("/kaggle/input")
if not IMAGE_DIR.is_dir() and KAGGLE_INPUT.is_dir():
    candidates = [
        p for p in KAGGLE_INPUT.rglob("gambar_terpilih")
        if p.is_dir()
    ]
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Tidak bisa menentukan folder gambar di /kaggle/input. "
            "Atur IMAGE_DIR secara eksplisit."
        )
    IMAGE_DIR = candidates[0]

if not CSV_PATH.is_file() and KAGGLE_INPUT.is_dir():
    candidates = list(KAGGLE_INPUT.rglob("gabungan_dengan_jumlah_sel.csv"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Tidak bisa menentukan CSV label di /kaggle/input. "
            "Atur CSV_PATH secara eksplisit."
        )
    CSV_PATH = candidates[0]

if Path("/kaggle/working").is_dir():
    OUTPUT_DIR = Path("/kaggle/working/Hasil/Chronological Order")

IMAGE_COLUMN = "Nama_Gambar"
TARGET_COLUMN = "Jumlah Sel (sel/mL)"

N_SPLITS = 5
TEST_RATIO = 0.20
EPOCHS = 15
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
SEED = 42

MODEL_NAMES = [
    "Custom CNN 29-Layer",
    "VGG16",
    "EfficientNet-B0",
    "Multilayer ANN",
    "ResNeXt50 32x4d",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Folder gambar tidak ditemukan: {IMAGE_DIR}")
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV label tidak ditemukan: {CSV_PATH}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = OUTPUT_DIR / "visualisasi"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Folder gambar: {IMAGE_DIR}")
print(f"File label: {CSV_PATH}")
print(f"Folder output: {OUTPUT_DIR}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

# ============================================================
# 2. Baca label dan cocokkan gambar dengan Nama_Gambar
# ============================================================

df = pd.read_csv(
    CSV_PATH,
    encoding="utf-8-sig",
    dtype={
        IMAGE_COLUMN: "string",
        TARGET_COLUMN: "string",
    },
)

required_columns = {IMAGE_COLUMN, TARGET_COLUMN}
if not required_columns.issubset(df.columns):
    raise KeyError(
        f"Kolom CSV tidak ditemukan: "
        f"{sorted(required_columns - set(df.columns))}"
    )

if df[IMAGE_COLUMN].isna().any():
    raise ValueError(f"Kolom {IMAGE_COLUMN} memiliki nilai kosong.")
if df[IMAGE_COLUMN].duplicated().any():
    raise ValueError(f"Kolom {IMAGE_COLUMN} memiliki nama duplikat.")

# Contoh "163.217.200" dikonversi menjadi 163217200.
target_text = df[TARGET_COLUMN].astype("string").str.strip()
thousands_format = target_text.str.fullmatch(
    r"\d{1,3}(?:\.\d{3})+"
).fillna(False)
target_text.loc[thousands_format] = target_text.loc[
    thousands_format
].str.replace(".", "", regex=False)

df["target_numeric"] = pd.to_numeric(target_text, errors="coerce")

if df["target_numeric"].isna().any():
    invalid_rows = df.loc[
        df["target_numeric"].isna(),
        [IMAGE_COLUMN, TARGET_COLUMN],
    ]
    raise ValueError(f"Label jumlah sel tidak valid:\n{invalid_rows}")

if not np.isfinite(df["target_numeric"].to_numpy(dtype=float)).all():
    raise ValueError("Target berisi nilai tak hingga.")

image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
image_files = [
    path
    for path in IMAGE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in image_extensions
]
image_map = {path.name: path for path in image_files}

if len(image_map) != len(image_files):
    raise ValueError("Ada nama file gambar duplikat di folder gambar.")

csv_names = set(df[IMAGE_COLUMN].astype(str))
image_names = set(image_map)

missing_images = sorted(csv_names - image_names)
unlabeled_images = sorted(image_names - csv_names)

if missing_images:
    raise FileNotFoundError(
        "Gambar dari CSV tidak ditemukan di folder gambar. "
        f"Contoh nama: {missing_images[:5]}"
    )
if unlabeled_images:
    raise ValueError(
        "Ada gambar tanpa label di CSV. "
        f"Contoh nama: {unlabeled_images[:5]}"
    )

df["image_path"] = df[IMAGE_COLUMN].map(
    lambda name: str(image_map[str(name)])
)

# Urut kronologis berdasarkan timestamp dalam nama gambar.
timestamp_text = df[IMAGE_COLUMN].str.extract(
    r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})",
    expand=False,
)
df["timestamp"] = pd.to_datetime(
    timestamp_text,
    format="%Y-%m-%d_%H-%M-%S",
    errors="coerce",
)

if df["timestamp"].isna().any():
    raise ValueError(
        "Ada nama gambar tanpa timestamp format "
        "YYYY-MM-DD_HH-MM-SS."
    )

df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
image_paths = df["image_path"].tolist()
targets = df["target_numeric"].to_numpy(dtype=np.float32)

print(f"Jumlah pasangan gambar-label: {len(df)}")
print(
    f"Rentang label: {targets.min():,.0f} sampai "
    f"{targets.max():,.0f} sel/mL"
)

# Pasangan label-gambar mengikuti Nama_Gambar seperti yang dikonfirmasi.
# Pastikan alignment tersebut memang benar sebelum menafsirkan hasil.


# ============================================================
# 3. Pisahkan 80% awal untuk pengembangan dan 20% akhir untuk test
# ============================================================

test_size = max(1, int(np.ceil(TEST_RATIO * len(targets))))
train_boundary = len(targets) - test_size

train_paths = image_paths[:train_boundary]
y_train = targets[:train_boundary]

test_paths = image_paths[train_boundary:]
y_test = targets[train_boundary:]
test_names = df[IMAGE_COLUMN].iloc[train_boundary:].to_numpy()

if len(y_train) <= N_SPLITS:
    raise ValueError("Data train tidak cukup untuk TimeSeriesSplit.")
if len(y_test) < 2:
    raise ValueError("Test set perlu minimal dua sampel untuk menghitung R².")

cv = TimeSeriesSplit(n_splits=N_SPLITS)
fold_splits = list(cv.split(np.arange(len(y_train))))

print(f"Data pengembangan/train: {len(y_train)}")
print(f"Data test final: {len(y_test)}")
print(f"Jumlah fold CV: {len(fold_splits)}")


# ============================================================
# 4. Dataset dan transformasi gambar
# ============================================================

class ImageRegressionDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = np.asarray(labels, dtype=np.float32)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image_tensor = self.transform(image.convert("RGB"))

        label = torch.tensor(self.labels[index], dtype=torch.float32)
        return image_tensor, label


imagenet_normalize = transforms.Normalize(
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
)


def make_transform(model_name):
    if model_name == "Multilayer ANN":
        image_size = 32
    elif model_name == "Custom CNN 29-Layer":
        image_size = 128
    else:
        image_size = 224

    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            imagenet_normalize,
        ]
    )


def make_loader(paths, labels_scaled, transform, shuffle):
    dataset = ImageRegressionDataset(paths, labels_scaled, transform)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


# ============================================================
# 5. Arsitektur lima model
# ============================================================

class CustomCNN29(nn.Module):
    """
    CNN custom: 13 Conv2d + 13 BatchNorm2d + 3 Linear
    = 29 layer berparameter.
    """

    def __init__(self):
        super().__init__()

        conv_config = [
            (3, 32), (32, 32),
            (32, 64), (64, 64),
            (64, 128), (128, 128), (128, 128),
            (128, 256), (256, 256), (256, 256),
            (256, 512), (512, 512), (512, 512),
        ]
        block_end_indices = {1, 3, 6, 9, 12}
        layers = []

        for index, (in_channels, out_channels) in enumerate(conv_config):
            layers.extend(
                [
                    nn.Conv2d(
                        in_channels,
                        out_channels,
                        kernel_size=3,
                        padding=1,
                        bias=False,
                    ),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU(inplace=True),
                ]
            )
            if index in block_end_indices:
                layers.append(nn.MaxPool2d(kernel_size=2))

        layers.extend(
            [
                nn.AdaptiveAvgPool2d((1, 1)),
                nn.Flatten(),
                nn.Linear(512, 256),
                nn.ReLU(),
                nn.Dropout(0.4),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, 1),
            ]
        )
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


class MultilayerANN(nn.Module):
    """ANN dengan input gambar RGB yang di-resize ke 32 x 32."""

    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.network(x)


class TransferRegressor(nn.Module):
    """Backbone ImageNet dibekukan; hanya head regresi yang dilatih."""

    def __init__(self, backbone, output_channels):
        super().__init__()
        self.backbone = backbone
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(output_channels, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

    def forward(self, x):
        features = self.backbone(x)
        return self.head(self.pool(features))


def make_model(model_name):
    if model_name == "Custom CNN 29-Layer":
        return CustomCNN29()

    if model_name == "Multilayer ANN":
        return MultilayerANN()

    if model_name == "VGG16":
        base = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        return TransferRegressor(base.features, output_channels=512)

    if model_name == "EfficientNet-B0":
        base = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )
        return TransferRegressor(base.features, output_channels=1280)

    if model_name == "ResNeXt50 32x4d":
        base = models.resnext50_32x4d(
            weights=models.ResNeXt50_32X4D_Weights.DEFAULT
        )
        backbone = nn.Sequential(*list(base.children())[:-2])
        return TransferRegressor(backbone, output_channels=2048)

    raise ValueError(f"Nama model tidak dikenal: {model_name}")


# ============================================================
# 6. Training dengan catatan loss train dan validation per epoch
# ============================================================

def evaluate_loss(model, loader, target_std):
    """
    Menghasilkan MAE pada target terstandardisasi dan dalam sel/mL.
    Validation dipakai untuk evaluasi saja, tidak untuk update bobot.
    """
    model.eval()
    absolute_error_scaled_sum = 0.0
    absolute_error_sum = 0.0
    number_of_samples = 0

    with torch.no_grad():
        for images, labels_scaled in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels_scaled = labels_scaled.to(DEVICE, non_blocking=True)

            predictions_scaled = model(images).squeeze(1)
            error_scaled = predictions_scaled - labels_scaled

            absolute_error_scaled_sum += torch.sum(
                torch.abs(error_scaled)
            ).item()
            absolute_error_sum += torch.sum(
                torch.abs(error_scaled * target_std)
            ).item()
            number_of_samples += len(images)

    return (
        absolute_error_scaled_sum / number_of_samples,
        absolute_error_sum / number_of_samples,
    )


def fit_one_run(
    model_name,
    fit_paths,
    fit_targets,
    validation_paths=None,
    validation_targets=None,
    seed=SEED,
):
    set_seed(seed)

    transform = make_transform(model_name)
    target_mean = float(np.mean(fit_targets))
    target_std = float(np.std(fit_targets))
    if target_std < 1e-8:
        target_std = 1.0

    fit_targets_scaled = (fit_targets - target_mean) / target_std
    fit_loader = make_loader(
        fit_paths,
        fit_targets_scaled,
        transform,
        shuffle=True,
    )
    fit_eval_loader = make_loader(
        fit_paths,
        fit_targets_scaled,
        transform,
        shuffle=False,
    )

    validation_loader = None
    if validation_paths is not None:
        validation_targets_scaled = (
            validation_targets - target_mean
        ) / target_std
        validation_loader = make_loader(
            validation_paths,
            validation_targets_scaled,
            transform,
            shuffle=False,
        )

    network = make_model(model_name).to(DEVICE)
    trainable_parameters = [
        parameter
        for parameter in network.parameters()
        if parameter.requires_grad
    ]
    optimizer = torch.optim.Adam(
        trainable_parameters,
        lr=LEARNING_RATE,
        weight_decay=1e-4,
    )
    loss_function = nn.L1Loss()
    history = []

    for epoch in range(1, EPOCHS + 1):
        network.train()

        # Backbone pretrained tetap dalam mode eval karena bobot dibekukan.
        if isinstance(network, TransferRegressor):
            network.backbone.eval()

        for images, labels_scaled in fit_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels_scaled = labels_scaled.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            predictions_scaled = network(images).squeeze(1)
            loss = loss_function(predictions_scaled, labels_scaled)
            loss.backward()
            optimizer.step()

        # Hitung ulang train loss pada mode eval agar dropout tidak menambah noise.
        train_loss, train_mae = evaluate_loss(
            network,
            fit_eval_loader,
            target_std,
        )

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_mae": train_mae,
        }

        if validation_loader is not None:
            validation_loss, validation_mae = evaluate_loss(
                network,
                validation_loader,
                target_std,
            )
            row["validation_loss"] = validation_loss
            row["validation_mae"] = validation_mae

        history.append(row)

    return network, target_mean, target_std, transform, history


def predict(model, paths, transform, target_mean, target_std):
    dummy_labels = np.zeros(len(paths), dtype=np.float32)
    loader = make_loader(paths, dummy_labels, transform, shuffle=False)

    model.eval()
    predictions = []

    with torch.no_grad():
        for images, _ in loader:
            images = images.to(DEVICE, non_blocking=True)
            prediction_scaled = model(images).squeeze(1)
            predictions.extend(prediction_scaled.cpu().numpy())

    return np.asarray(predictions, dtype=np.float32) * target_std + target_mean


def calculate_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }


# ============================================================
# 7. CV 5-fold: semua model, catat loss setiap epoch
# ============================================================

cv_epoch_rows = []
cv_fold_metric_rows = []
cv_prediction_rows = []

for model_index, model_name in enumerate(MODEL_NAMES):
    print(f"\n===== CV model: {model_name} =====")

    for fold_number, (train_indices, validation_indices) in enumerate(
        fold_splits,
        start=1,
    ):
        print(f"\nFold {fold_number}/{N_SPLITS}")

        fold_train_paths = [train_paths[i] for i in train_indices]
        fold_validation_paths = [train_paths[i] for i in validation_indices]
        fold_train_targets = y_train[train_indices]
        fold_validation_targets = y_train[validation_indices]

        network, target_mean, target_std, transform, history = fit_one_run(
            model_name=model_name,
            fit_paths=fold_train_paths,
            fit_targets=fold_train_targets,
            validation_paths=fold_validation_paths,
            validation_targets=fold_validation_targets,
            seed=SEED + model_index * 100 + fold_number,
        )

        for item in history:
            cv_epoch_rows.append(
                {
                    "Model": model_name,
                    "Fold": fold_number,
                    **item,
                }
            )

        validation_predictions = predict(
            network,
            fold_validation_paths,
            transform,
            target_mean,
            target_std,
        )

        fold_metrics = calculate_metrics(
            fold_validation_targets,
            validation_predictions,
        )
        cv_fold_metric_rows.append(
            {
                "Model": model_name,
                "Fold": fold_number,
                **fold_metrics,
            }
        )

        for index, actual_value, predicted_value in zip(
            validation_indices,
            fold_validation_targets,
            validation_predictions,
        ):
            cv_prediction_rows.append(
                {
                    "Model": model_name,
                    "Fold": fold_number,
                    "Nama_Gambar": df[IMAGE_COLUMN].iloc[index],
                    "actual": actual_value,
                    "predicted": predicted_value,
                    "residual": actual_value - predicted_value,
                }
            )

        del network
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


# ============================================================
# 8. Simpan CSV riwayat epoch dan skor CV
# ============================================================

cv_epoch_df = pd.DataFrame(cv_epoch_rows)
cv_fold_metrics_df = pd.DataFrame(cv_fold_metric_rows)
cv_predictions_df = pd.DataFrame(cv_prediction_rows)

cv_epoch_summary_df = (
    cv_epoch_df.groupby(["Model", "epoch"], as_index=False)
    .agg(
        train_loss_mean=("train_loss", "mean"),
        train_loss_std=("train_loss", "std"),
        validation_loss_mean=("validation_loss", "mean"),
        validation_loss_std=("validation_loss", "std"),
        train_mae_mean=("train_mae", "mean"),
        train_mae_std=("train_mae", "std"),
        validation_mae_mean=("validation_mae", "mean"),
        validation_mae_std=("validation_mae", "std"),
    )
)

cv_summary_df = (
    cv_fold_metrics_df.groupby("Model", as_index=False)
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
        n_folds=("Fold", "nunique"),
    )
    .sort_values("MAE_mean")
    .reset_index(drop=True)
)

cv_epoch_df.to_csv(
    OUTPUT_DIR / "dl_cv_loss_per_fold_per_epoch.csv",
    index=False,
    encoding="utf-8-sig",
)
cv_epoch_summary_df.to_csv(
    OUTPUT_DIR / "dl_cv_loss_mean_per_epoch.csv",
    index=False,
    encoding="utf-8-sig",
)
cv_fold_metrics_df.to_csv(
    OUTPUT_DIR / "dl_cv_metrics_per_fold.csv",
    index=False,
    encoding="utf-8-sig",
)
cv_predictions_df.to_csv(
    OUTPUT_DIR / "dl_cv_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)
cv_summary_df.to_csv(
    OUTPUT_DIR / "dl_cv_model_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nRingkasan hasil CV:")
display(cv_summary_df.round(3))


# ============================================================
# 9. Visualisasi MAE train dan validation rata-rata per model
# ============================================================

for model_name in MODEL_NAMES:
    curve = cv_epoch_summary_df[
        cv_epoch_summary_df["Model"] == model_name
    ].sort_values("epoch")

    train_std = curve["train_mae_std"].fillna(0).to_numpy()
    validation_std = curve["validation_mae_std"].fillna(0).to_numpy()
    epochs = curve["epoch"].to_numpy()

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(
        epochs,
        curve["train_mae_mean"],
        marker="o",
        label="Train MAE (rata-rata 5 fold)",
    )
    ax.fill_between(
        epochs,
        curve["train_mae_mean"].to_numpy() - train_std,
        curve["train_mae_mean"].to_numpy() + train_std,
        alpha=0.18,
    )

    ax.plot(
        epochs,
        curve["validation_mae_mean"],
        marker="o",
        label="Validation MAE (rata-rata 5 fold)",
    )
    ax.fill_between(
        epochs,
        curve["validation_mae_mean"].to_numpy() - validation_std,
        curve["validation_mae_mean"].to_numpy() + validation_std,
        alpha=0.18,
    )

    ax.set_title(f"Train vs Validation MAE — {model_name}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MAE (sel/mL)")
    ax.set_xticks(range(1, EPOCHS + 1))
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    safe_name = model_name.lower().replace(" ", "_").replace("-", "_")
    fig.savefig(
        OUTPUT_DIR / f"mae_cv_{safe_name}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


# ============================================================
# 10. Fit final semua model pada seluruh 80%, evaluasi test sekali
# ============================================================

best_model_name = cv_summary_df.iloc[0]["Model"]
final_history_rows = []
final_metric_rows = []
test_predictions = {}
best_checkpoint = None

for model_index, model_name in enumerate(MODEL_NAMES):
    print(f"\n===== Final fit pada 80%: {model_name} =====")

    network, target_mean, target_std, transform, history = fit_one_run(
        model_name=model_name,
        fit_paths=train_paths,
        fit_targets=y_train,
        validation_paths=None,
        validation_targets=None,
        seed=SEED + 1000 + model_index,
    )

    for item in history:
        final_history_rows.append(
            {
                "Model": model_name,
                **item,
            }
        )

    train_predictions = predict(
        network,
        train_paths,
        transform,
        target_mean,
        target_std,
    )
    test_predictions_for_model = predict(
        network,
        test_paths,
        transform,
        target_mean,
        target_std,
    )
    test_predictions[model_name] = test_predictions_for_model

    final_metric_rows.append(
        {
            "Model": model_name,
            "Data": "Train",
            **calculate_metrics(y_train, train_predictions),
        }
    )
    final_metric_rows.append(
        {
            "Model": model_name,
            "Data": "Test",
            **calculate_metrics(y_test, test_predictions_for_model),
        }
    )

    if model_name == best_model_name:
        best_checkpoint = {
            "model_name": model_name,
            "model_state_dict": {
                key: value.detach().cpu()
                for key, value in network.state_dict().items()
            },
            "target_mean": target_mean,
            "target_std": target_std,
            "seed": SEED + 1000 + model_index,
        }

    del network
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# ============================================================
# 11. Simpan hasil final train/test ke CSV
# ============================================================

final_history_df = pd.DataFrame(final_history_rows)
final_metrics_df = pd.DataFrame(final_metric_rows)

test_prediction_df = pd.DataFrame(
    {
        "Nama_Gambar": test_names,
        "actual": y_test,
    }
)

for model_name, predictions in test_predictions.items():
    safe_name = model_name.lower().replace(" ", "_").replace("-", "_")
    test_prediction_df[f"predicted_{safe_name}"] = predictions
    test_prediction_df[f"residual_{safe_name}"] = y_test - predictions

final_history_df.to_csv(
    OUTPUT_DIR / "dl_final_train_loss_per_epoch.csv",
    index=False,
    encoding="utf-8-sig",
)
final_metrics_df.to_csv(
    OUTPUT_DIR / "dl_final_metrics_train_test.csv",
    index=False,
    encoding="utf-8-sig",
)
test_prediction_df.to_csv(
    OUTPUT_DIR / "dl_test_predictions_all_models.csv",
    index=False,
    encoding="utf-8-sig",
)

torch.save(
    best_checkpoint,
    OUTPUT_DIR / "dl_best_model_by_cv.pth",
)

print("\nEvaluasi final semua model:")
display(
    final_metrics_df
    .pivot(index="Model", columns="Data", values=["MAE", "RMSE", "R2"])
    .round(3)
)
print(f"Model terpilih berdasarkan rata-rata MAE CV: {best_model_name}")


# ============================================================
# 12. Visualisasi test: satu gambar per model
# ============================================================

formatter_ribuan = FuncFormatter(
    lambda value, position: f"{value:,.0f}".replace(",", ".")
)

for model_name in MODEL_NAMES:
    predicted = test_predictions[model_name]
    residual = y_test - predicted

    mae = mean_absolute_error(y_test, predicted)
    rmse = np.sqrt(mean_squared_error(y_test, predicted))
    r2 = r2_score(y_test, predicted)

    lower = min(0.0, float(y_test.min()), float(predicted.min()))
    upper = max(float(y_test.max()), float(predicted.max()))

    fig = plt.figure(figsize=(12, 6))
    grid = fig.add_gridspec(
        2,
        2,
        height_ratios=[1, 0.22],
        hspace=0.4,
        wspace=0.25,
    )

    ax_residual = fig.add_subplot(grid[0, 0])
    ax_actual_predicted = fig.add_subplot(grid[0, 1])
    ax_metrics = fig.add_subplot(grid[1, :])
    ax_metrics.axis("off")

    ax_residual.scatter(predicted, residual, alpha=0.8)
    ax_residual.axhline(
        0,
        color="red",
        linestyle="--",
        label="Residual = 0",
    )
    ax_residual.set_title("Residual — Test")
    ax_residual.set_xlabel("Predicted jumlah sel")
    ax_residual.set_ylabel("Residual (Actual - Predicted)")
    ax_residual.xaxis.set_major_formatter(formatter_ribuan)
    ax_residual.yaxis.set_major_formatter(formatter_ribuan)
    ax_residual.legend()
    ax_residual.grid(True, alpha=0.3)

    ax_actual_predicted.scatter(y_test, predicted, alpha=0.8, label="Prediksi")
    ax_actual_predicted.plot(
        [lower, upper],
        [lower, upper],
        color="red",
        linestyle="--",
        label="Prediksi sempurna",
    )
    ax_actual_predicted.set_xlim(lower, upper)
    ax_actual_predicted.set_ylim(lower, upper)
    ax_actual_predicted.set_title("Actual vs Predicted — Test")
    ax_actual_predicted.set_xlabel("Actual jumlah sel")
    ax_actual_predicted.set_ylabel("Predicted jumlah sel")
    ax_actual_predicted.xaxis.set_major_formatter(formatter_ribuan)
    ax_actual_predicted.yaxis.set_major_formatter(formatter_ribuan)
    ax_actual_predicted.legend()
    ax_actual_predicted.grid(True, alpha=0.3)

    # Tandai nilai actual test di bawah 100 juta.
    for index in np.flatnonzero(y_test < 100_000_000):
        ax_actual_predicted.annotate(
            f"{y_test[index]:,.0f}".replace(",", "."),
            (y_test[index], predicted[index]),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=8,
        )

    ax_metrics.text(
        0.5,
        0.5,
        f"MAE = {mae:,.0f}     |     "
        f"RMSE = {rmse:,.0f}     |     "
        f"R² = {r2:.3f}",
        ha="center",
        va="center",
        fontsize=11,
        transform=ax_metrics.transAxes,
        bbox={
            "boxstyle": "round,pad=0.5",
            "facecolor": "white",
            "edgecolor": "gray",
        },
    )

    fig.suptitle(f"Hasil Evaluasi Test — {model_name}", fontsize=14)
    fig.subplots_adjust(
        left=0.08,
        right=0.97,
        top=0.86,
        bottom=0.08,
    )

    safe_name = model_name.lower().replace(" ", "_").replace("-", "_")
    fig.savefig(
        PLOT_DIR / f"{safe_name}_evaluasi_test.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


# ============================================================
# 13. Tampilkan dan simpan ringkasan metrik test
# ============================================================

test_metrics_df = pd.DataFrame(
    [
        {
            "Model": model_name,
            **calculate_metrics(y_test, test_predictions[model_name]),
        }
        for model_name in MODEL_NAMES
    ]
)

test_metrics_df.to_csv(
    OUTPUT_DIR / "dl_test_metrics_all_models.csv",
    index=False,
    encoding="utf-8-sig",
)

display(test_metrics_df.round(3))

print(f"\nHasil CSV dan model tersimpan di: {OUTPUT_DIR}")
print(f"Visualisasi tersimpan di: {PLOT_DIR}")